# cAIuldron v2.0 — Main App

**Run All Cells** to launch the Gradio interface at `http://127.0.0.1:7860`

**Tech Stack:**
- 🔍 Groq Llama 3.2 Vision — ingredient detection
- 🥗 USDA Database — nutrition lookup
- 📚 LangChain + ChromaDB RAG — similar recipe search
- 🌐 Tavily — web recipe inspiration
- 🍳 Groq Llama 3.3 70B — recipe generation
- 🖼️ HF FLUX.1-schnell — dish image generation
- 📊 LangSmith — pipeline observability
- 🤖 LangGraph — agent orchestration

In [1]:
# ── Load all modules ───────────────────────────────────────────────────────
%run 1_config.ipynb
%run 2_tools.ipynb
%run 3_rag.ipynb
%run 4_agent.ipynb

print('\n✅ All modules loaded — ready to launch Gradio')

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
Environment loaded
Project root : d:\Github\UF PROJECT\cAIuldron
Recipes JSON : True → d:\Github\UF PROJECT\cAIuldron\data\processed\recipes\full_recipes.json
Nutrition DB : True → d:\Github\UF PROJECT\cAIuldron\data\nutrition_lookup_full.json
ChromaDB dir : d:\Github\UF PROJECT\cAIuldron\app_v2\.chroma_db
✅ All API keys loaded
   LangSmith tracing: true
✅ Config complete
   Vision model : meta-llama/llama-4-scout-17b-16e-instruct
   Text model   : llama-3.3-70b-versatile
   Image model  : black-forest-labs/FLUX.1-schnell
   HF URL       : https://router.huggingface.co/hf-inference/models/black-forest-labs/FLUX.1-schnell
✅ Tool imports OK
✅ Vision tool ready
✅ Nutrition DB loaded: 525 ingredients
✅ Nutrition tool ready
✅ Web search tool ready
✅ Image generation tool ready
   Model: black-forest-labs/FLUX.1-schnell
   Note: install nest_asyncio if running in Jupyter → pip install nest_asyncio
✅ All tools loaded — u

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ ChromaDB loaded: 7913 recipes (skipping rebuild)
✅ RAG collection ready: "caiuldron_recipes"
✅ retrieve_similar_recipes() ready
   Test query returned 3 results
   Top result: "Lemon Chicken" (similarity: 0.785)
✅ Agent imports OK
✅ RecipeState defined
✅ ChatGroq ready: llama-3.3-70b-versatile
✅ Node 1: detect_ingredients_node
✅ Node 2: estimate_nutrition_node
✅ Node 3a: rag_search_node
✅ Node 3b: web_search_node
✅ Node 4: generate_recipes_node
✅ Node 5: generate_images_node
✅ LangGraph compiled
   Nodes: detect_ingredients → estimate_nutrition
          → [rag_search || web_search] → generate_recipes → generate_images → END

✅ All modules loaded — ready to launch Gradio


In [2]:
import os
import gradio as gr
from io import BytesIO
from PIL import Image as PILImage

print('✅ Gradio ready')

✅ Gradio ready


In [3]:
def process_image(image: PILImage.Image, progress=gr.Progress()):
    """Main pipeline entry point called by Gradio submit button."""
    if image is None:
        empty = '### Upload an ingredient photo to begin.'
        return empty, empty, empty, empty, [], empty

    progress(0.05, desc='Preparing image...')

    # Convert PIL → bytes
    buf = BytesIO()
    image.convert('RGB').save(buf, format='JPEG', quality=85)
    image_bytes = buf.getvalue()

    # Build initial state
    initial_state: RecipeState = {
        'image_bytes':              image_bytes,
        'detected_ingredients':     [],
        'ingredient_detection_raw': '',
        'nutrition_per_ingredient': {},
        'nutrition_summary':        '',
        'rag_retrieved_recipes':    [],
        'rag_context_block':        '',
        'web_search_results':       [],
        'web_context_block':        '',
        'recipes':                  [],
        'errors':                   [],
        'processing_time_seconds':  {},
    }

    progress(0.1, desc='Running AI pipeline (vision → RAG → recipes → images)...')

    # Run full LangGraph pipeline
    final = recipe_graph.invoke(initial_state)

    progress(0.95, desc='Formatting results...')

    # ── Ingredients Tab ────────────────────────────────────────────────────
    ingredients   = final.get('detected_ingredients', [])
    errors        = final.get('errors', [])

    ing_md = '## 🔍 Detected Ingredients\n\n'
    ing_md += '\n'.join(f'- {ing}' for ing in ingredients) if ingredients else '_No ingredients detected._'
    if errors:
        ing_md += '\n\n> **Warnings**: ' + '; '.join(errors)

    # ── Nutrition Tab ──────────────────────────────────────────────────────
    nut_md = '## 🥗 Nutrition Estimates\n\n'
    nut_md += final.get('nutrition_summary', '_No nutrition data._')

    # ── RAG Tab ────────────────────────────────────────────────────────────
    rag_recipes = final.get('rag_retrieved_recipes', [])
    rag_md = '## 📚 Similar Recipes from Our Database\n\n'
    if rag_recipes:
        for r in rag_recipes:
            rag_md += (
                f"**{r['title']}**  \n"
                f"Cuisine: {r['cuisine'].title()} | Difficulty: {r['difficulty'].title()} | "
                f"Time: {r['cooking_time_minutes']} min | Similarity: {r['similarity_score']}\n\n"
            )
    else:
        rag_md += '_No similar recipes found in database._'

    # ── Recipes Tab ────────────────────────────────────────────────────────
    recipes = final.get('recipes', [])
    recipe_blocks = []
    gallery_images = []

    for i, r in enumerate(recipes):
        block = (
            f"## {i+1}. {r['title']}\n\n"
            f"**Cuisine**: {r['cuisine']} | **Difficulty**: {r['difficulty'].title()}  \n"
            f"**Prep**: {r['prep_time_minutes']} min | **Cook**: {r['cook_time_minutes']} min | "
            f"**Total**: {r['total_time_minutes']} min | **Serves**: {r['servings']}\n\n"
            f"### Ingredients\n"
            + '\n'.join(f'- {ing}' for ing in r['ingredients'])
            + '\n\n### Instructions\n'
            + '\n'.join(f"{j+1}. {step}" for j, step in enumerate(r['instructions']))
        )
        if r.get('rag_source_titles'):
            block += f"\n\n_Inspired by: {', '.join(r['rag_source_titles'][:2])}_"
        recipe_blocks.append(block)

        img = bytes_to_pil(r.get('image_bytes'))
        if img is not None:
            gallery_images.append(img)

    recipe_md = '\n\n---\n\n'.join(recipe_blocks) if recipe_blocks else '_No recipes generated._'

    # ── Pipeline Stats Tab ─────────────────────────────────────────────────
    timing = final.get('processing_time_seconds', {})
    total  = sum(timing.values())
    stats_md = '## 📊 Pipeline Timing\n\n'
    stats_md += f'**Total**: {total:.1f}s\n\n'
    for stage, t in timing.items():
        stats_md += f'- {stage.replace("_", " ").title()}: {t:.2f}s\n'
    if os.environ.get('LANGCHAIN_TRACING_V2') == 'true':
        stats_md += '\n\n[View trace in LangSmith →](https://smith.langchain.com/)'

    progress(1.0, desc='Done!')
    return ing_md, nut_md, rag_md, recipe_md, gallery_images, stats_md

print('✅ process_image() ready')

✅ process_image() ready


In [4]:
_CSS = """
/* 讓所有容器自動撐高，不截斷內容 */
.gradio-container,
.tabs > div,
.tabitem,
div[role="tabpanel"],
.block,
.grid-wrap,
.gallery {
    overflow: visible !important;
    height: auto !important;
    max-height: none !important;
}

/* 隱藏空的 gallery 格子（灰色框） */
.gallery button:not(:has(img)),
.gallery .thumbnail:not(:has(img)),
.gallery li:not(:has(img)) {
    display: none !important;
}

/* 全域隱藏滾輪 */
::-webkit-scrollbar { display: none !important; }
* { scrollbar-width: none !important; }

/* 進度條 */
.generating {
    position: relative !important;
    font-size: 0.85em;
    color: #f97316;
    padding: 4px 0;
}
"""

# ── Gradio UI ──────────────────────────────────────────────────────────────
with gr.Blocks(title='cAIuldron v2.0', css=_CSS) as demo:

    gr.Markdown("""
    # 🍳 cAIuldron v2.0
    **AI Recipe Generator** — Upload a photo of your ingredients and get 5 personalized recipes,
    complete with nutrition info, dish images, and web-inspired variations.

    *Groq Vision · Groq 70B · ChromaDB RAG · LangGraph · Tavily · FLUX.1 · LangSmith*
    """)

    with gr.Row(equal_height=False):

        # ── Left: Upload + Info ────────────────────────────────────────────
        with gr.Column(scale=1, min_width=280):
            gr.Markdown('### 📤 Upload Ingredient Photo')
            image_input = gr.Image(
                type='pil',
                label='Photo of Ingredients',
                height=280,
            )
            submit_btn = gr.Button('🚀 Generate Recipes', variant='primary', size='lg')
            status_out = gr.Markdown(value='', visible=True)

            gr.Markdown("""
            **How it works:**
            1. Upload a clear photo of raw ingredients
            2. Groq Vision identifies ingredients
            3. RAG searches 7,913-recipe database
            4. Tavily searches web for inspiration
            5. Groq 70B generates 5 unique recipes
            6. FLUX.1 renders a dish photo per recipe

            **Powered by:**
            - Groq API (Vision + Text)
            - LangGraph (Agent orchestration)
            - ChromaDB (Recipe RAG)
            - Tavily (Web search)
            - HF FLUX.1-schnell (Images)
            - LangSmith (Observability)
            """)

        # ── Right: Results Tabs ────────────────────────────────────────────
        with gr.Column(scale=2):
            with gr.Tabs():

                with gr.Tab('🔍 Ingredients'):
                    ingredients_out = gr.Markdown(
                        value='Upload a photo and click **Generate Recipes** to begin.'
                    )

                with gr.Tab('🥗 Nutrition'):
                    nutrition_out = gr.Markdown(
                        value='Nutrition data will appear after ingredient detection.'
                    )

                with gr.Tab('📚 Similar Recipes (RAG)'):
                    rag_out = gr.Markdown(
                        value='Top 3 similar recipes from our 7,913-recipe ChromaDB database.'
                    )

                with gr.Tab('🍳 Generated Recipes'):
                    recipes_out = gr.Markdown(
                        value='Your 5 personalized recipes will appear here.'
                    )

                with gr.Tab('🖼️ Dish Images'):
                    gallery_out = gr.Gallery(
                        label='AI-Generated Dish Photos (FLUX.1)',
                        columns=3,
                        object_fit='cover',
                    )

                with gr.Tab('📊 Pipeline Stats'):
                    stats_out = gr.Markdown(
                        value='Per-node timing and LangSmith trace link will appear here.'
                    )

    submit_btn.click(
        fn=process_image,
        inputs=[image_input],
        outputs=[ingredients_out, nutrition_out, rag_out, recipes_out, gallery_out, stats_out],
        show_progress='minimal',
    )

    gr.Markdown("""
    ---
    **cAIuldron v2.0** · Built with LangGraph + ChromaDB RAG + LangSmith
    Data: USDA FoodData Central · RecipeNLG Dataset (7,913 recipes)
    """)

print('✅ Gradio UI built')

C:\Users\Champion\AppData\Local\Temp\ipykernel_40440\3175720609.py:36: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(title='cAIuldron v2.0', css=_CSS) as demo:


✅ Gradio UI built


In [5]:
# ── Launch ─────────────────────────────────────────────────────────────────
# share=True generates a public gradio.live URL (valid 72h) for sharing demos
demo.launch(
    server_name='127.0.0.1',
    server_port=7860,
    share=False,     # Change to True to get a public URL
    inbrowser=True,
)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


  Generating 5 dish images via FLUX.1-schnell...
  ✅ Asian-Style Grilled Chicken and Fruit Skewers
  ✅ Mediterranean Chicken and Fruit Salad
  ✅ American-Style Strawberry Watermelon Chicken Salad
  ✅ Mexican-Style Grilled Chicken and Fruit Tacos
  ✅ Italian-Style Chicken and Cabbage Salad with Fruit
  Generating 5 dish images via FLUX.1-schnell...
  ✅ Asian-Style Grilled Fish and Vegetables
  ✅ Mediterranean Chicken and Vegetable Kabobs
  ✅ American-Style Strawberry Kiwi Melon Salad with Grilled Chicken
  ✅ Mexican-Style Grilled Pork Chops with Avocado Salsa
  ✅ Italian-Style Chicken and Vegetable Stir-Fry with Hazelnuts
  Generating 5 dish images via FLUX.1-schnell...
  ✅ Asian-Style Grilled Fish with Melon Salsa
  ✅ Mediterranean Chicken and Cabbage Salad
  ✅ American-Style Strawberry Melon Shortcake
  ✅ Mexican-Style Grilled Pork Chop with Fruit Salsa
  ✅ Italian-Style Chicken and Cabbage Frittata
  Generating 5 dish images via FLUX.1-schnell...
  Image gen status 402: {"error":"You